+++
title = "DataFrame support"
menu = "main"
weight = 8
toc = true
aliases = ["dataframes", "polars"]
+++

Prince accepts pandas and Polars DataFrames in every estimator that consumes two-dimensional tabular data: PCA, CA, MCA, FAMD, MFA, and PGA. [Narwhals](https://narwhals-dev.github.io/narwhals/) provides the DataFrame compatibility layer. Prince converts these inputs to pandas DataFrames and NumPy arrays internally, so pandas remains a required dependency.

Polars is optional. Install it alongside Prince if you want to use Polars DataFrames:

```sh
pip install prince polars
```

## Polars input

Pass a Polars DataFrame directly to `fit`, `transform`, or `fit_transform`. This works with PCA, CA, MCA, FAMD, MFA, and PGA.

In [1]:
from __future__ import annotations

import polars as pl

import prince

X = pl.DataFrame(
    {
        "height": [1.61, 1.72, 1.80, 1.68, 1.75],
        "weight": [54.0, 68.0, 81.0, 63.0, 75.0],
        "age": [23.0, 35.0, 29.0, 41.0, 32.0],
        "score": [7.2, 8.1, 8.8, 7.7, 8.4],
    }
)
X

height,weight,age,score
f64,f64,f64,f64
1.61,54.0,23.0,7.2
1.72,68.0,35.0,8.1
1.8,81.0,29.0,8.8
1.68,63.0,41.0,7.7
1.75,75.0,32.0,8.4


Prince returns a pandas DataFrame by default. This preserves the output behavior of earlier Prince versions. Because Polars and NumPy do not have an index, their default result uses a pandas `RangeIndex`. A pandas input keeps its original index.

In [2]:
coordinates = prince.PCA(n_components=2, random_state=42).fit_transform(X)
print(type(coordinates))
coordinates

<class 'pandas.DataFrame'>


component,0,1
0,2.850243,-1.114482
1,-0.190770,0.480283
2,-2.278634,-0.825293
3,0.743605,1.621341
4,-1.124443,-0.161849


## Polars output

Use scikit-learn's [`set_output`](https://scikit-learn.org/stable/auto_examples/miscellaneous/plot_set_output.html) API when the result should also be a Polars DataFrame.

In [3]:
pca = prince.PCA(n_components=2, random_state=42).set_output(transform="polars")
coordinates_polars = pca.fit_transform(X)
print(type(coordinates_polars))
coordinates_polars

<class 'polars.dataframe.frame.DataFrame'>


0,1
f64,f64
2.850243,-1.114482
-0.19077,0.480283
-2.278634,-0.825293
0.743605,1.621341
-1.124443,-0.161849


## NumPy input

PCA also accepts a two-dimensional NumPy array. Array input has no column names, and the default pandas result uses a `RangeIndex`.

In [4]:
X_numpy = X.to_numpy()
pca = prince.PCA(n_components=2, random_state=42).fit(X_numpy)
coordinates = pca.transform(X_numpy)
print(type(coordinates))
coordinates

<class 'pandas.DataFrame'>


component,0,1
0,2.850243,-1.114482
1,-0.190770,0.480283
2,-2.278634,-0.825293
3,0.743605,1.621341
4,-1.124443,-0.161849


## NumPy output

Pass `as_array=True` to PCA's `transform`, `fit_transform`, or `inverse_transform` method to return a NumPy array instead of the default pandas DataFrame.

In [5]:
coordinates_numpy = pca.transform(X_numpy, as_array=True)
print(type(coordinates_numpy))
coordinates_numpy

<class 'numpy.ndarray'>


array([[ 2.85024267, -1.11448151],
       [-0.19077026,  0.48028256],
       [-2.27863386, -0.82529274],
       [ 0.74360471,  1.62134058],
       [-1.12444326, -0.1618489 ]])

## Estimator coverage

Polars input is supported by PCA, CA, MCA, FAMD, MFA, and PGA. GPA is different: it consumes a three-dimensional array with shape `(n_shapes, n_points, n_dimensions)`, which cannot be represented as a two-dimensional pandas or Polars DataFrame. GPA therefore continues to accept array-like input.